# Pipeline — Hackathón PNUD: riesgo de desplazamiento por desastres

Notebook reproducible (entregable 2). Genera `outputs/entregas/entrega_*.csv` a partir de los CSV del reto.
La lógica está en `src/hackaton/`; este notebook configura el entorno y la ejecuta.

**Cómo reproducirlo**
1. Tener el repositorio completo: en Colab, clonarlo en `/content/Hackathon-pnud-riesgo-desplazamiento` (o usar la copia del equipo en Google Drive); en local, abrir este notebook desde la carpeta del repositorio.
2. Copiar `entrenamiento.csv`, `prueba_equipos.csv` y `prueba_oculta.csv` en `data/reto/`.
3. Ejecutar todas las celdas. La entrega queda en `outputs/entregas/` y se valida contra el formato oficial antes de guardarse.

**Declaración de uso de IA generativa (T&C 9.1).** Herramienta: Claude Code (Anthropic), modelo Claude Opus 5.5 (`claude-opus-5-5`). Alcance: lectura de las reglas, propuesta de alternativas de modelado, y escritura y depuración del código (carga y tratamiento de datos, exploración, réplica de la métrica, validador, variables, entrenamiento y validación). La lógica de modelado, la selección de variables y la interpretación de resultados las dirige y verifica el equipo. Detalle completo en `DECLARACION_IA.md`.

## 1. Entorno (Colab o local)

In [ ]:
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
CANDIDATAS = [Path("/content/Hackathon-pnud-riesgo-desplazamiento")]              # repositorio descomprimido en Colab
if EN_COLAB:
    !pip install -q catboost
    if not (CANDIDATAS[0] / "src" / "hackaton").exists():   # si no está, usar la copia del equipo en Drive
        from google.colab import drive
        drive.mount("/content/drive")
        CANDIDATAS.append(Path("/content/drive/MyDrive/Documentos Juan Felipe/Hackaton_PENUD"))
CANDIDATAS += [Path.cwd(), Path.cwd().parent]                # ejecución local
RAIZ = next((p for p in CANDIDATAS if (p / "src" / "hackaton").exists()), None)
assert RAIZ is not None, "No encuentro el repositorio: clónalo en /content/Hackathon-pnud-riesgo-desplazamiento o ajusta CANDIDATAS."

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))
print("Raíz del proyecto:", RAIZ)

## 2. Datos

El día D: copiar `entrenamiento.csv`, `prueba_equipos.csv` y `prueba_oculta.csv` a `data/reto/`.
Para practicar: `DATOS = "sinteticos"` (generarlos con `python scripts/generar_datos_sinteticos.py`).

In [ ]:
DATOS = "reto"      # "reto" el día D · "sinteticos" para practicar
SOLUCION = "A"      # "A" o "B"
EXTRA = ""          # p. ej. "--solo-validacion"

from hackaton import config
from hackaton.datos import cargar_reto

datos = cargar_reto(config.DIR_RETO if DATOS == "reto" else config.DIR_SINTETICOS)
for nombre, df in datos.items():
    print(f"{nombre:15s} {len(df):>7,} filas")
print("Tasa de eventos en entrenamiento:", round(datos["entrenamiento"]["tiene_evento"].mean(), 3))

## 3. Validación temporal → modelo final → entrega validada
Imprime por corte las métricas oficiales y el puntaje de validación, y en una segunda línea los diagnósticos.

In [ ]:
%run scripts/entrenar.py --solucion {SOLUCION} --datos {DATOS} {EXTRA}

## 4. Bitácora de experimentos
Una fila por corte y corrida. Se elige el modelo por el **puntaje de validación** promedio de F1–F3 (fórmula oficial sobre nuestros cortes, no la nota oficial); los diagnósticos ayudan a entender.

In [ ]:
import pandas as pd
pd.read_csv(config.DIR_REPORTES / "experimentos.csv").tail(12)

## 5. Geovisor (entregable 3)
Genera `outputs/geovisor/geovisor.html` con la entrega más reciente.

In [ ]:
%run scripts/generar_geovisor.py --modelo "Solución {SOLUCION}"